In [3]:
import os
import numpy as np


RAW_DIR = "Datasets/Raw"
OUT_DIR = "Datasets"
DEFAULT_OUT_NAME = "Main.npz"


def list_subjects():
    if not os.path.exists(RAW_DIR):
        raise FileNotFoundError(f"Missing folder: {RAW_DIR}")

    return sorted([
        d for d in os.listdir(RAW_DIR)
        if os.path.isdir(os.path.join(RAW_DIR, d))
    ])


def list_npz_files(subject):
    subject_dir = os.path.join(RAW_DIR, subject)

    return sorted([
        os.path.join(subject_dir, f)
        for f in os.listdir(subject_dir)
        if f.endswith(".npz")
    ])


def load_subject_data(subject):
    files = list_npz_files(subject)

    X_raw_all = []
    X_filtered_all = []
    X_csp_all = []
    y_all = []
    subject_all = []
    session_all = []

    for file_path in files:
        data = np.load(file_path, allow_pickle=True)

        if "X_raw" not in data or "X_filtered" not in data or "X_csp" not in data or "y" not in data:
            print(f"Skipping invalid file: {file_path}")
            continue

        X_raw = data["X_raw"]
        X_filtered = data["X_filtered"]
        X_csp = data["X_csp"]
        y = data["y"]

        n = len(y)

        X_raw_all.append(X_raw)
        X_filtered_all.append(X_filtered)
        X_csp_all.append(X_csp)
        y_all.append(y)

        subject_all.extend([subject] * n)
        session_all.extend([os.path.basename(file_path).replace(".npz", "")] * n)

    if not y_all:
        return None

    return {
        "X_raw": np.concatenate(X_raw_all, axis=0),
        "X_filtered": np.concatenate(X_filtered_all, axis=0),
        "X_csp": np.concatenate(X_csp_all, axis=0),
        "y": np.concatenate(y_all, axis=0),
        "subjects": np.array(subject_all),
        "sessions": np.array(session_all),
    }


def choose_subjects(subjects):
    print("\nAvailable subjects:")
    for i, subject in enumerate(subjects, start=1):
        print(f"{i}. {subject}")

    print("\nChoose:")
    print("A = all subjects")
    print("Example specific users: 1,3,4")

    choice = input("\nSelection: ").strip().upper()

    if choice == "A" or choice == "":
        return subjects

    indices = [int(x.strip()) - 1 for x in choice.split(",")]
    return [subjects[i] for i in indices if 0 <= i < len(subjects)]


def balanced_trial_limit(X_raw, X_filtered, X_csp, y, subjects, sessions, max_trials_per_class):
    left_idx = np.where(y == 1)[0]
    right_idx = np.where(y == 2)[0]

    max_possible = min(len(left_idx), len(right_idx))

    if max_trials_per_class is None or max_trials_per_class > max_possible:
        max_trials_per_class = max_possible

    left_keep = left_idx[:max_trials_per_class]
    right_keep = right_idx[:max_trials_per_class]

    keep = np.concatenate([left_keep, right_keep])
    np.random.shuffle(keep)

    return (
        X_raw[keep],
        X_filtered[keep],
        X_csp[keep],
        y[keep],
        subjects[keep],
        sessions[keep],
        max_possible,
        max_trials_per_class,
    )


def main():
    subjects = list_subjects()

    if not subjects:
        print("No subjects found.")
        return

    selected_subjects = choose_subjects(subjects)

    print("\nSelected subjects:")
    for s in selected_subjects:
        print("-", s)

    merged = []

    for subject in selected_subjects:
        subject_data = load_subject_data(subject)

        if subject_data is None:
            print(f"No valid data for subject: {subject}")
            continue

        merged.append(subject_data)

    if not merged:
        print("No valid datasets found.")
        return

    X_raw = np.concatenate([m["X_raw"] for m in merged], axis=0)
    X_filtered = np.concatenate([m["X_filtered"] for m in merged], axis=0)
    X_csp = np.concatenate([m["X_csp"] for m in merged], axis=0)
    y = np.concatenate([m["y"] for m in merged], axis=0)
    trial_subjects = np.concatenate([m["subjects"] for m in merged], axis=0)
    trial_sessions = np.concatenate([m["sessions"] for m in merged], axis=0)

    left_count = int((y == 1).sum())
    right_count = int((y == 2).sum())
    max_balanced = min(left_count, right_count)

    print("\nMerged total:")
    print("X_raw:", X_raw.shape)
    print("X_filtered:", X_filtered.shape)
    print("X_csp:", X_csp.shape)
    print("y:", y.shape)
    print("Left trials:", left_count)
    print("Right trials:", right_count)
    print("Max balanced trials per class:", max_balanced)
    print("Max total balanced trials:", max_balanced * 2)

    trial_input = input(
        f"\nTrials per class to use [Enter = max {max_balanced}]: "
    ).strip()

    if trial_input == "":
        trials_per_class = None
    else:
        trials_per_class = int(trial_input)

    (
        X_raw,
        X_filtered,
        X_csp,
        y,
        trial_subjects,
        trial_sessions,
        max_possible,
        used_per_class,
    ) = balanced_trial_limit(
        X_raw,
        X_filtered,
        X_csp,
        y,
        trial_subjects,
        trial_sessions,
        trials_per_class,
    )

    out_name = input(f"\nOutput file name [Enter = {DEFAULT_OUT_NAME}]: ").strip()

    if out_name == "":
        out_name = DEFAULT_OUT_NAME

    if not out_name.endswith(".npz"):
        out_name += ".npz"

    os.makedirs(OUT_DIR, exist_ok=True)
    out_path = os.path.join(OUT_DIR, out_name)

    np.savez(
        out_path,
        X_raw=X_raw,
        X_filtered=X_filtered,
        X_csp=X_csp,
        y=y,
        subjects=trial_subjects,
        sessions=trial_sessions,
        selected_subjects=np.array(selected_subjects),
        trials_per_class=used_per_class,
        max_possible_trials_per_class=max_possible,
        label_left=1,
        label_right=2,
    )

    print("\nSaved merged dataset:")
    print(out_path)
    print("\nFinal:")
    print("X_raw:", X_raw.shape)
    print("X_filtered:", X_filtered.shape)
    print("X_csp:", X_csp.shape)
    print("y:", y.shape)
    print("Left:", int((y == 1).sum()))
    print("Right:", int((y == 2).sum()))




if __name__ == "__main__":
    main()


Available subjects:
1. ARNAV
2. ARNAV copy
3. DEVIN
4. ERIM
5. FAUZAAN

Choose:
A = all subjects
Example specific users: 1,3,4

Selected subjects:
- ARNAV

Merged total:
X_raw: (570, 8, 1750)
X_filtered: (570, 8, 1750)
X_csp: (570, 8, 750)
y: (570,)
Left trials: 285
Right trials: 285
Max balanced trials per class: 285
Max total balanced trials: 570

Saved merged dataset:
Datasets\Main.npz

Final:
X_raw: (570, 8, 1750)
X_filtered: (570, 8, 1750)
X_csp: (570, 8, 750)
y: (570,)
Left: 285
Right: 285
